# 02 — Stationarity and Cointegration: Gold, Silver, and the Macro-Financial System

This notebook builds on the data prepared in `01_Data_Exploration.ipynb` (log-prices,
log-ratio, macro variables, train/val/test split) and on the `stationarity.py` and
`cointegration.py` modules.

**Objective**: characterize the order of integration of the series, then test for a
long-run cointegrating relationship — first in the bivariate gold-silver system
(Part I), then in the four-variable macro-financial system (Part II) — both
statically (training sample) and dynamically (temporal stability).


# PART I. Bivariate System (Gold-Silver)

## 1. Loading data and modules

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from preprocessing import train_val_test_split
from stationarity import stationarity_table
from cointegration import (
    engle_granger_test,
    ols_cointegration_residual,
    johansen_all_specs,
    johansen_monthly_robustness,
    rolling_cointegration,
)
from utils import find_significant_intervals

In [ ]:
df = pd.read_csv("../data/processed/gold_silver_processed.csv", index_col=0, parse_dates=True)

df_train, df_val, df_test = train_val_test_split(df, train_frac=0.70, val_frac=0.85)

print(f"Training sample: {df_train.index.min().date()} -> {df_train.index.max().date()} "
      f"({len(df_train)} observations)")

df_train.head()

## 2. Stationarity tests

The ADF (Augmented Dickey-Fuller) and KPSS (Kwiatkowski-Phillips-Schmidt-Shin) tests
are applied in level and in first differences on the training sample. The two tests
are complementary: ADF tests H0 = unit root, KPSS tests H0 = stationarity. Agreement
between the two conclusions strengthens the diagnosis.

### 2.1 Level tests

In [ ]:
level_table = stationarity_table(df_train[["log_gold", "log_silver", "log_ratio"]])
level_table

### 2.2 First-difference tests

In [ ]:
diff_table = stationarity_table(df_train[["dlog_gold", "dlog_silver"]])
diff_table

### 2.3 Conclusion on the order of integration

Gold and silver log-prices, as well as the log-ratio, are not stationary in level: ADF
does not reject H0 (unit root) and KPSS rejects H0 (stationarity) — the two tests agree.

In first differences (logarithmic returns), ADF rejects H0 and KPSS does not reject
it: the series become stationary. Log-prices are therefore integrated of order one,
**I(1)**, which is the necessary condition for the cointegration tests that follow.

## 3. Bivariate gold-silver cointegration analysis

### 3.1 Rationale

A linear combination of two I(1) series that is itself stationary would indicate a
long-run equilibrium relationship (cointegration) between the two assets: the series
can diverge in the short run, but a correction mechanism pulls them back toward a
common equilibrium.

Two approaches are used below as a cross-check:
- the **Engle-Granger** test (`statsmodels` implementation, MacKinnon critical values);
- a manual OLS regression followed by an ADF test on the residual, to illustrate the
  mechanics of the test and verify consistency of the conclusions.

In [ ]:
eg_result = engle_granger_test(df_train["log_gold"], df_train["log_silver"])
eg_result

In [ ]:
alpha_hat, beta_hat, residual, adf_res = ols_cointegration_residual(
    df_train["log_gold"], df_train["log_silver"]
)

print("Estimated long-run relation (OLS):")
print(f"log_gold = {alpha_hat:.4f} + {beta_hat:.4f} * log_silver")
print()
print("ADF test on the residual (H0 = unit root, i.e. no cointegration):")
print(f"  ADF stat  : {adf_res['ADF stat']:.4f}")
print(f"  p-value   : {adf_res['p-value']:.4f}")
print(f"  crit. 5%  : {adf_res['crit_5%']:.4f}")

Both approaches agree: the Engle-Granger test and the ADF test on the OLS residual
fail to reject the null hypothesis of no cointegration on the training sample.

### 3.2 Johansen test: robustness across specifications

The Johansen test does not rely on an arbitrary choice of dependent variable (unlike
Engle-Granger) and allows testing several cointegration ranks simultaneously. It is
applied here across several specifications (with/without deterministic trend,
different lag orders) to assess the robustness of the result.

In [ ]:
johansen_global_specs = johansen_all_specs(
    data=df_train,
    variables=["log_gold", "log_silver"],
    det_orders=(0, 1),
    k_ar_diffs=(0, 1, 2),
    signif_index=1,  # 5% threshold
)

johansen_global_specs

Across all specifications tested, the trace statistic does not exceed the 5% critical
value for H0: r=0. The Johansen test thus confirms the absence of cointegration
detected by Engle-Granger on the training sample, regardless of specification.

## 4. Complementary analysis of temporal stability

The global tests above are run on a single window (the training sample) and may mask
a cointegrating relationship that only holds over certain sub-periods. Three
complementary analyses are therefore conducted over the full 2000-2024 period. They
are not used to calibrate a predictive model, but to interpret possible variations in
the long-run relationship across economic regimes.

### 4.1 Johansen test by sub-period

In [ ]:
periods = {
    "pre_crisis_global": df.loc["2000":"2007-06"],
    "global_financial_crisis": df.loc["2007-07":"2009"],
    "post_crisis_QE": df.loc["2010":"2014"],
    "commodity_stress": df.loc["2015":"2016"],
    "late_cycle_expansion": df.loc["2017":"2019"],
    "covid_shock": df.loc["2020":"2022"],
}

johansen_period_specs = johansen_all_specs(
    data=periods,
    variables=["log_gold", "log_silver"],
    det_orders=(0, 1),
    k_ar_diffs=(0, 1, 2),
    signif_index=1,
)

johansen_period_specs

### 4.2 Rolling Engle-Granger cointegration

In [ ]:
rolling_eg = rolling_cointegration(
    df, target="log_gold", features=["log_silver"], window=1000
)

rolling_eg.head()

In [ ]:
intervals_df = find_significant_intervals(rolling_eg["p_value"], threshold=0.05, min_days=60)

intervals_df

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(rolling_eg.index, rolling_eg["p_value"], label="Rolling Engle-Granger p-value")
plt.axhline(0.05, linestyle="--", color="red", label="5% threshold")

for _, row in intervals_df.iterrows():
    plt.axvspan(row["Start"], row["End"], alpha=0.15, color="steelblue")

plt.title("Rolling Engle-Granger Cointegration Test — Gold-Silver")
plt.ylabel("p-value")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

The shaded areas correspond to windows where the p-value stays below 5% for more
than 60 consecutive days: these are transient episodes where a local cointegrating
relationship is detected, notably around the 2015-2016 commodity stress episode and
the COVID-19 shock. These episodes are too short and unstable to qualify as a
long-run relationship in the strict sense, but they motivate the localized VECM
estimated in the next notebook.

### 4.3 Robustness to observation frequency: Johansen on monthly data

The Johansen test is replicated on monthly-aggregated data (last observed price of
the month) to verify that the conclusions obtained at daily frequency are not merely
an artifact of data granularity.

In [ ]:
df_m = df[["gold", "silver"]].resample("ME").last().dropna()
df_m["log_gold"] = np.log(df_m["gold"])
df_m["log_silver"] = np.log(df_m["silver"])

periods_m = {name: df_m.loc[p.index.min():p.index.max()] for name, p in periods.items()}

results_df_m = johansen_monthly_robustness(
    periods=periods_m,
    variables=["log_gold", "log_silver"],
    det_orders=(0, 1),
    k_ar_diffs=(0, 1, 2),
    signif_index=1,
    min_obs=36,
)

results_df_m

In [ ]:
summary_m = (
    results_df_m
    .dropna(subset=["Cointégration"])
    .groupby("Période")
    .agg(nb_tests=("Cointégration", "count"), nb_cointegration=("Cointégration", "sum"))
)
summary_m["proportion_cointegration"] = summary_m["nb_cointegration"] / summary_m["nb_tests"]

for period, row in summary_m.iterrows():
    print(
        f"{period}: {row['nb_cointegration']} out of {row['nb_tests']} specifications "
        f"detect cointegration ({row['proportion_cointegration']:.0%})"
    )

## 5. Conclusion — Part I

The bilateral relationship between gold and silver does not appear to be a stable
cointegrating relationship over the full 2000-2024 period.

The global Engle-Granger and Johansen tests, run on the training sample, fail to
robustly reject the null hypothesis of no cointegration — a conclusion confirmed by
the manual OLS regression.

However, the dynamic and sub-period analyses reveal transient episodes of
cointegration, in particular during the 2015-2016 commodity stress episode and the
COVID-19 shock. These results suggest that the long-run relationship between the two
metals depends strongly on the economic regime and market conditions rather than
being a stable, permanent property of the system.

This instability has two implications for the rest of the study:
- **Bivariate framework**: a VAR in differences (rather than a global VECM) is
  preferred for short-run dynamics (`03_Bivariate_VAR_VECM.ipynb`); a VECM is only
  estimated locally, on the cointegration episode identified in section 4.2.
- **Multivariate framework**: the instability of the bivariate relationship motivates
  extending the system to macro-financial factors (DXY, US 10-year yield) that could
  jointly explain the co-movement of both metals — tested formally in Part II below.

---

# PART II. Multivariate Macro-Financial System (Gold, Silver, DXY, US10Y)

Part I found no robust global cointegration between gold and silver alone. This part
tests whether introducing the US dollar index and the US 10-year yield — two common
macro-financial drivers documented in the literature (Baur & Lucey, 2010; Bampinas &
Panagiotidis, 2015) — changes this diagnosis, following exactly the same structure as
Part I: stationarity, static cointegration, then temporal stability.

## 6. Loading multivariate data

In [ ]:
df_macro = pd.read_csv(
    "../data/processed/gold_silver_macro_processed.csv", index_col=0, parse_dates=True
)

df_macro_train, df_macro_val, df_macro_test = train_val_test_split(
    df_macro, train_frac=0.70, val_frac=0.85
)

print(f"Training sample: {df_macro_train.index.min().date()} -> "
      f"{df_macro_train.index.max().date()} ({len(df_macro_train)} observations)")

df_macro_train.head()

## 7. Stationarity tests — multivariate system

Same logic as Part I, applied to the four-variable system: `log_gold`, `log_silver`,
`log_dxy`, and `us10y` (kept in level rather than log, see `preprocessing.add_log_returns`
note — a rate is not log-transformed like a price).

### 7.1 Level tests

In [ ]:
level_table_macro = stationarity_table(
    df_macro_train[["log_gold", "log_silver", "log_dxy", "us10y"]]
)
level_table_macro

### 7.2 First-difference tests

In [ ]:
diff_table_macro = stationarity_table(
    df_macro_train[["dlog_gold", "dlog_silver", "dlog_dxy", "dus10y"]]
)
diff_table_macro

### 7.3 Conclusion on the order of integration

As in the bivariate case, all four level series fail to reject the unit-root
hypothesis (ADF) while KPSS rejects stationarity — consistent I(1) behavior. Their
first differences are stationary under both tests. The four-variable system is
therefore I(1), the necessary condition for the multivariate cointegration tests
below — this is the result that notebook `04_Multivariate_VAR_VECM.ipynb` refers to
as "established in the multivariate stationarity notebook".

## 8. Multivariate cointegration analysis

### 8.1 Rationale

With more than two variables, `engle_granger_test` (which wraps `statsmodels.coint`,
limited to two series) can no longer be used directly. The Engle-Granger logic is
generalized manually: `log_gold` is regressed by OLS on `log_silver`, `log_dxy`, and
`us10y`, and a generic ADF test is applied to the residual.

**Methodological caveat**: unlike the bivariate case, this residual ADF test does not
use MacKinnon's cointegration-specific critical values (they are not defined for more
than two variables) — generic ADF critical values are used instead, which tend to
reject the null of no cointegration too easily. This test is therefore indicative,
not a substitute for the Johansen test that follows.

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

X = sm.add_constant(df_macro_train[["log_silver", "log_dxy", "us10y"]])
y = df_macro_train["log_gold"]

ols_multi = sm.OLS(y, X).fit()
residual_multi = ols_multi.resid

adf_stat, adf_pvalue, *_ = adfuller(residual_multi)

print("Estimated long-run relation (OLS, multivariate):")
print(f"log_gold = {ols_multi.params['const']:.4f} "
      f"+ {ols_multi.params['log_silver']:.4f} * log_silver "
      f"+ {ols_multi.params['log_dxy']:.4f} * log_dxy "
      f"+ {ols_multi.params['us10y']:.4f} * us10y")
print()
print("ADF test on the residual (generic critical values, H0 = unit root):")
print(f"  ADF stat : {adf_stat:.4f}")
print(f"  p-value  : {adf_pvalue:.4f}")

### 8.2 Johansen test: robustness across specifications

In [ ]:
johansen_global_specs_multi = johansen_all_specs(
    data=df_macro_train,
    variables=["log_gold", "log_silver", "log_dxy", "us10y"],
    det_orders=(0, 1),
    k_ar_diffs=(0, 1, 2),
    signif_index=1,  # 5% threshold
)

johansen_global_specs_multi

*(Fill in once run: compare directly against §8.1 — this is exactly the
Engle-Granger-vs-Johansen divergence discussed in the project report, and worth
naming explicitly if the two disagree: Engle-Granger tests a single equation, while
Johansen imposes a full-system structure that loses power when the relationship is
regime-dependent rather than stable over the whole training window.)*

## 9. Complementary analysis of temporal stability — multivariate system

Same three complementary analyses as Part I, section 4, run on the four-variable
system, using the same six macro regimes for direct comparability with the bivariate
results.

### 9.1 Johansen test by sub-period

In [ ]:
johansen_period_specs_multi = johansen_all_specs(
    data={name: p for name, p in periods.items()},
    variables=["log_gold", "log_silver", "log_dxy", "us10y"],
    det_orders=(0, 1),
    k_ar_diffs=(0, 1, 2),
    signif_index=1,
)

johansen_period_specs_multi

### 9.2 Rolling Engle-Granger cointegration (multivariate)

In [ ]:
rolling_multi = rolling_cointegration(
    df_macro, target="log_gold", features=["log_silver", "log_dxy", "us10y"], window=1000
)

rolling_multi.head()

In [ ]:
intervals_multi_df = find_significant_intervals(
    rolling_multi["p_value"], threshold=0.05, min_days=60
)

intervals_multi_df

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(rolling_multi.index, rolling_multi["p_value"], label="Rolling Engle-Granger p-value (multivariate)")
plt.axhline(0.05, linestyle="--", color="red", label="5% threshold")

for _, row in intervals_multi_df.iterrows():
    plt.axvspan(row["Start"], row["End"], alpha=0.15, color="steelblue")

plt.title("Rolling Engle-Granger Cointegration Test — Gold, Silver, DXY, US10Y")
plt.ylabel("p-value")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

*(Fill in once run: name the longest interval explicitly — expected to be the
Jan 2013 - Sep 2017 episode used for the multivariate VECM in notebook 04 — and
contrast the number/length of episodes here against the bivariate rolling result in
§4.2: more numerous and longer episodes here is the central evidence for the "macro
factors explain part of the bivariate instability" claim.)*

**Methodological note**: as in the bivariate case, `find_significant_intervals`
reports the span of consecutive significant windows — but because the rolling window
(1000 observations) shifts one day at a time, consecutive windows overlap by 999
observations out of 1000. A long detected episode therefore reflects a smoothed,
highly autocorrelated signal by construction, not day-by-day independent confirmation
of cointegration — worth stating explicitly rather than treating episode length as a
precise, independent measurement.

### 9.3 Robustness to observation frequency: Johansen on monthly data

In [ ]:
df_m_macro = df_macro[["gold", "silver", "dxy", "us10y"]].resample("ME").last().dropna()
df_m_macro["log_gold"] = np.log(df_m_macro["gold"])
df_m_macro["log_silver"] = np.log(df_m_macro["silver"])
df_m_macro["log_dxy"] = np.log(df_m_macro["dxy"])

periods_m_macro = {
    name: df_m_macro.loc[p.index.min():p.index.max()] for name, p in periods.items()
}

results_df_m_macro = johansen_monthly_robustness(
    periods=periods_m_macro,
    variables=["log_gold", "log_silver", "log_dxy", "us10y"],
    det_orders=(0, 1),
    k_ar_diffs=(0, 1, 2),
    signif_index=1,
    min_obs=36,
)

results_df_m_macro

In [ ]:
summary_m_macro = (
    results_df_m_macro
    .dropna(subset=["Cointégration"])
    .groupby("Période")
    .agg(nb_tests=("Cointégration", "count"), nb_cointegration=("Cointégration", "sum"))
)
summary_m_macro["proportion_cointegration"] = (
    summary_m_macro["nb_cointegration"] / summary_m_macro["nb_tests"]
)

for period, row in summary_m_macro.iterrows():
    print(
        f"{period}: {row['nb_cointegration']} out of {row['nb_tests']} specifications "
        f"detect cointegration ({row['proportion_cointegration']:.0%})"
    )

## 10. Conclusion — Part II

*(Fill in once run, following the same structure as §5: state whether the global
Engle-Granger/Johansen divergence found in §8 holds up, summarize which sub-periods
and how many rolling episodes are detected here versus in the bivariate case in Part
I, and close on the same forward-looking note Part I ends on — that these episodes,
notably 2013-2017, motivate the localized multivariate VECM estimated in
`04_Multivariate_VAR_VECM.ipynb`.)*

**Overall comparison to keep in mind between Part I and Part II**: introducing DXY and
US10Y is expected to produce more numerous and longer-lived rolling cointegration
episodes than the bivariate case, and a significant global Engle-Granger result even
where Johansen remains non-significant — evidence that part of the bivariate
instability reflects the omission of common macro-financial factors, rather than a
genuine absence of any long-run link between the two metals.